# Phase 5d - gaussian_gray amplitude sweep
gaussian_gray is coherent + untinted but a weaker count-forcer than the tinted bump (the color DC was doing some object-seeding). Amplitude is a free knob: sweep `gray_amp` to find where the untinted bump matches plain gaussian's count control while keeping images clean.

**Runtime:** GPU (~15 min).

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os, yaml, torch
import matplotlib.pyplot as plt
from src.prompts import build_prompt
from src.pipeline import load_sdxl
from src.noise_layout import count_aware_latent
from src.detector import Detector
from src.scoring import count_from_detections
from src.config import load_config

In [ ]:
cfg = load_config('configs/phase5d.yaml')
raw = yaml.safe_load(open('configs/phase5d.yaml'))
amps = raw['gray_amps']; obj = cfg.objects[0]
pk = dict(omega=raw['omega'], alpha=raw['alpha'], fill=raw['box_fill'])
pipe = load_sdxl(); det = Detector()
SS = pipe.unet.config.sample_size
def cnt(img):
    return count_from_detections(det.detect(img, [obj]), obj, cfg.score_threshold)
def base_latent(seed):
    g = torch.Generator(device='cpu').manual_seed(seed)
    z = torch.randn((1, pipe.unet.config.in_channels, SS, SS), generator=g)
    return z.to(pipe.device, pipe.dtype)
def gen_latent(prompt, lat):
    return pipe(prompt, latents=lat, num_inference_steps=cfg.num_inference_steps).images[0]

In [ ]:
# Neutral gray latent direction (amp=1), scaled per-amp in the loop.
@torch.no_grad()
def gray_direction(probe=2.0):
    vae = pipe.vae; orig = next(vae.parameters()).dtype
    vae.to(torch.float32); sf = vae.config.scaling_factor
    try:
        z0 = torch.zeros(1, 4, 16, 16, device=pipe.device, dtype=torch.float32)
        rgb0 = vae.decode(z0 / sf).sample.mean(dim=(0, 2, 3))
        cols = []
        for ch in range(4):
            z = z0.clone(); z[:, ch] += probe
            cols.append((vae.decode(z / sf).sample.mean(dim=(0, 2, 3)) - rgb0) / probe)
        M = torch.stack(cols).cpu()
    finally:
        vae.to(orig)
    if not torch.isfinite(M).all():
        return torch.ones(4)
    a = torch.linalg.pinv(M.t()) @ torch.ones(3)
    return a / a.norm() * (4 ** 0.5)
gray0 = gray_direction()
print('gray direction (amp=1):', [round(x, 3) for x in gray0.tolist()])

In [ ]:
rows = []
for N in cfg.counts:
    prompt = build_prompt(N, obj)
    for seed in cfg.seeds:
        base = base_latent(seed)
        rows.append({'N': N, 'seed': seed, 'amp': 0.0,
                     'rendered': cnt(gen_latent(prompt, base.clone()))})
        for amp in amps:
            lat = count_aware_latent(base, N, scheme='gaussian_gray',
                                     channel_weights=gray0 * amp, **pk)
            rows.append({'N': N, 'seed': seed, 'amp': amp,
                         'rendered': cnt(gen_latent(prompt, lat))})
    print(f'N={N} done')
df = pd.DataFrame(rows)
df['correct'] = df['rendered'] == df['N']
os.makedirs('results', exist_ok=True)
df.to_csv('results/phase5d_counts.csv', index=False)
df.assign(e=(df['N'] - df['rendered']).abs()).groupby('amp').agg(
    exact_acc=('correct', 'mean'), mae=('e', 'mean'))

In [ ]:
# Accuracy & MAE vs gray amplitude (amp=0 is baseline).
g = df.assign(e=(df['N'] - df['rendered']).abs()).groupby('amp')
acc, mae = g['correct'].mean(), g['e'].mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(acc.index, acc.values, 'o-'); axes[0].set_xlabel('gray_amp (0=baseline)')
axes[0].set_ylabel('exact-count accuracy'); axes[0].set_ylim(0, 1)
axes[0].set_title('Accuracy vs amplitude'); axes[0].axhline(0.44, color='r', ls=':', label='plain gaussian 0.44')
axes[0].legend(fontsize=8)
axes[1].plot(mae.index, mae.values, 'o-'); axes[1].set_xlabel('gray_amp (0=baseline)')
axes[1].set_ylabel('MAE'); axes[1].set_title('MAE vs amplitude'); axes[1].axhline(1.68, color='r', ls=':', label='plain gaussian 1.68')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig('results/phase5d_sweep.png', dpi=100, bbox_inches='tight'); plt.show()
print('acc:\n', acc, '\nMAE:\n', mae)

In [ ]:
# Eyeball: baseline vs gray at each amplitude (asked N0). Coherent + count?
N0, seed0 = 6 if 6 in cfg.counts else cfg.counts[0], cfg.seeds[0]
prompt0 = build_prompt(N0, obj); base = base_latent(seed0)
cells = [('baseline', base.clone())] + \
        [(f'amp={a}', count_aware_latent(base, N0, scheme='gaussian_gray',
                                         channel_weights=gray0 * a, **pk)) for a in amps]
fig, axes = plt.subplots(1, len(cells), figsize=(3.0 * len(cells), 3.3))
for ax, (name, lat) in zip(axes, cells):
    im = gen_latent(prompt0, lat)
    ax.imshow(im); ax.axis('off'); ax.set_title(f'{name} | asked {N0}->{cnt(im)}', fontsize=8)
plt.tight_layout()
plt.savefig('results/phase5d_eyeball.png', dpi=90, bbox_inches='tight'); plt.show()

## How to read this
- **Accuracy rises with gray_amp toward (or past) plain gaussian's 0.44, and the eyeball stays coherent/untinted** = the clean capstone: a training-free, quality-preserving, high-count mitigation. Pick the amp at the accuracy peak that still looks natural.
- **Accuracy plateaus below gaussian, or the eyeball degrades at high amp** = there's a real coherence/accuracy tradeoff; report the best gray_amp as the quality-preserving option and plain gaussian as the max-accuracy option.
- Watch for **too-high amp** re-introducing artifacts (over-strong injection).